# Chapter 10 Lab — GPU-Aware HPA + Canary Rollout Simulation

**Estimated time:** 20 minutes · **Read-and-run.** Read/modify a Kubernetes HPA policy driven by
a custom GPU metric, compute desired replicas, and simulate a canary traffic split.

In [ ]:
hpa = {
  'minReplicas': 2, 'maxReplicas': 10,
  'metric': 'gpu_utilization', 'target': 60,  # target 60% GPU util
}
def desired_replicas(current_replicas, current_metric, hpa):
    # Kubernetes HPA formula: ceil(current * (metric / target)), clamped.
    import math
    want = math.ceil(current_replicas * (current_metric / hpa['target']))
    return max(hpa['minReplicas'], min(hpa['maxReplicas'], want))
for util in (30, 60, 95, 140):
    print(f'GPU util {util:3d}%  (from 4 replicas) -> desired {desired_replicas(4, util, hpa)}')

In [ ]:
# Canary rollout: split traffic stable vs canary and tally.
import random; random.seed(1)
def canary_route(n, canary_pct):
    tally={'stable':0,'canary':0}
    for _ in range(n):
        tally['canary' if random.random() < canary_pct/100 else 'stable'] += 1
    return tally
print('10% canary over 1000 requests:', canary_route(1000, 10))
print('Promote by raising canary_pct (10 -> 50 -> 100) while watching error/latency SLOs.')

## Key takeaway
Effective LLM serving ops need **GPU-aware autoscaling signals** (not just CPU) and **staged
rollouts** (canary / blue-green) sized to the risk of the change.